In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt

from kuramoto.config import (
    SimulationConfig,
    GridConfig,
    CouplingConfig,
    InitThetaConfig,
    InitOmegaConfig,
    KernelComponentConfig,
    build_simulation,
)
from kuramoto.coupling import apply_node_lesions, plot_lesioned_coupling
from kuramoto.simulation import KuramotoParams
from kuramoto.analysis import (
    get_R, 
    compute_effective_coupling, 
    avg_effective_coupling,
    functional_connectivity,
    get_R_link,
)
from kuramoto.adjoint import (
    get_adjoint_grads,
    get_adjoint_metrics,
    plot_adjoint_metrics,
)
from kuramoto.plotting import (
    plot_2d, plot_coupling_matrix, set_plot_settings,
    plot_single_metric_rt_comparison,
    plot_r_vs_lesion_frac,
    plot_auc_abc_diagram,
    plot_auc_dotplot,
    plot_abc_lollipop,
    plot_abc_spread_boxplot_h,
    color_for_metric,
)
from kuramoto.network import (
    create_cortical_graph,
    plot_graph_metrics,
    get_graph_metrics,
    plot_metric,
)
from kuramoto.experiments import (
    evaluate_single_lesion,
    evaluate_metric_scores,
    aggregate_scores,
    run_lesion_study,
)

from jax import numpy as jnp

set_plot_settings()

SEED = 42
grid_shape = (12, 12)
N = grid_shape[0] * grid_shape[1]
T_END = 10.0
dt = 0.01

RNG = np.random.default_rng(SEED)


n_rows, n_cols = grid_shape

group_ids = np.zeros((n_rows, n_cols), dtype=int)
group_ids[n_rows // 2 :, :] = 1 # Top half is group 1, bottom half is group 0
group_ids[n_rows // 2 -2: n_rows // 2 +2, n_cols // 2 -2: n_cols // 2 +2] = 2 # 3x3 block in center is group 2
group_ids = group_ids.ravel().tolist()

components = [
    KernelComponentConfig(
        kernel="gaussian",
        base_strength=1.0,
        kernel_params={"sigma": 1.0},
        radius=2.0,
        node_groups=[0],
        edge_mode="within",
    ),
    KernelComponentConfig(
        kernel="gaussian",
        base_strength=1.0,
        kernel_params={"sigma": 1.0},
        radius=2.0,
        node_groups=[1],
        edge_mode="within",
    ),
    KernelComponentConfig( # Fully connected coupling from group 2 to groups 0 and 1
        kernel="gaussian",
        base_strength=0.8,
        kernel_params={"sigma": 4.0},
        radius=4.0,
        node_groups=[2],
        edge_mode="outgoing",
    ),
    KernelComponentConfig( 
        kernel="gaussian",
        base_strength=0.8,
        kernel_params={"sigma": 4.0},
        radius=4.0,
        node_groups=[2],
        edge_mode="incoming",
    ),
    KernelComponentConfig( # weak one way coupling from group 1 to group 0
        kernel="gaussian",
        base_strength=0.2,
        kernel_params={"sigma": 2.0},
        radius=2.0,
        node_groups=[1],
        edge_mode="custom",
        to_node_groups=[0],
    ),
]

cfg = SimulationConfig(
    grid=GridConfig(shape=grid_shape, periodic=False),
    coupling=CouplingConfig(
        kernel="gaussian",
        base_strength=1.0,
        radius=4.0,
        mode="spatial",
        components=components,
        group_ids=group_ids,
    ),
    initial_theta=InitThetaConfig(mode="uniform"),
    initial_omega=InitOmegaConfig(mode="normal", mu=0.0, sigma=0.3),
    seed=42,
)

sim = build_simulation(config=cfg, rng=np.random.default_rng(SEED))

# Run base simulation
res_base = sim.run((0, T_END), dt, rng=RNG)

# Create graphs
G = create_cortical_graph(sim)

K_eff_avg = avg_effective_coupling(sim.results["theta"], sim.coupling.K)
G_eff = create_cortical_graph(K_eff_avg, omega=sim.params.omega)

C_avg = functional_connectivity(sim.results["theta"], dt=dt)
G_C_avg = create_cortical_graph(C_avg, omega=sim.params.omega)

In [ ]:
graph_metrics = get_graph_metrics(G)
graph_metrics_eff = get_graph_metrics(G_eff)
graph_metrics_C_avg = get_graph_metrics(G_C_avg)

# plot_graph_metrics(graph_metrics,grid_shape=grid_shape,title="Base coupling network metrics")
# plot_graph_metrics(graph_metrics_eff,grid_shape=grid_shape,title="Effective coupling network metrics")
# plot_graph_metrics(graph_metrics_C_avg,grid_shape=grid_shape,title="Functional connectivity network metrics")

### 2) Adjoint metrics

In [ ]:
adj_metrics = get_adjoint_metrics(sim=sim, T_END=T_END, dt=dt, n_ig_steps=20)
# fig, axs = plot_adjoint_metrics(
#     adj_metrics,
#     grid_shape=grid_shape,
#     n_ig_steps=20,
#     title="Adjoint lesion metrics: linearized (top) vs Integrated Gradients (bottom)",
# )
# plt.show()

## Lesion Studies

In [ ]:
# Assemble all metrics into a single dictionary
metrics = {
    "deg_base": graph_metrics["deg_cent"],
    "deg_eff": graph_metrics_eff["deg_cent"],
    "closeness_base": graph_metrics["closeness"],
    "closeness_eff": graph_metrics_eff["closeness"],
    "betweenness_base": graph_metrics["betweenness"],
    "betweenness_eff": graph_metrics_eff["betweenness"],
    "eigenvector_base": graph_metrics["eigenvector"],
    "eigenvector_eff": graph_metrics_eff["eigenvector"],
    "eigenvector_C_avg": graph_metrics_C_avg["eigenvector"],
    "IRm_a": adj_metrics["IRm_a"],
    "IG_IRm_a": adj_metrics["IG_IRm_a"],
    "IRlink_a": adj_metrics["IRlink_a"],
    "IG_IRlink_a": adj_metrics["IG_IRlink_a"],
}

N_RANDOM_REPEATS = 10

K_base = sim.coupling.K

### Single metric lesion study

In [ ]:
metric = graph_metrics_C_avg["eigenvector"]
metric_name = "eigenvector_C_avg"

plot_metric(metric, grid_shape, "Eigenvector centrality")

lesion_frac = 0.10
lesion_strength = 1.0

# Lesion top x% of nodes by metric
res_ranked_lesion, alpha_ranked, K_lesioned_ranked = evaluate_single_lesion(sim, metric, lesion_frac, lesion_strength, T_END, dt, RNG)

# Lesion x% of nodes randomly (N_RANDOM_REPEATS times)
R_random_lesion = np.zeros((len(res_base['ts']), N_RANDOM_REPEATS))
for i in range(N_RANDOM_REPEATS):
    res_random_lesion, alpha_random, K_lesioned_random = evaluate_single_lesion(sim, "random", lesion_frac, lesion_strength, T_END, dt, RNG)
    R_random_lesion[:, i] = get_R(res_random_lesion["theta"])[0]
R_random_lesion_mean = np.mean(R_random_lesion, axis=1)
R_random_lesion_std = np.std(R_random_lesion, axis=1)

# Show lesioning pattern
fig, axs = plt.subplots(2, 2, figsize=(10, 10))
plot_lesioned_coupling(alpha_ranked, K_base, K_lesioned_ranked, grid_shape, axs=axs[0, :])
plot_lesioned_coupling(alpha_random, K_base, K_lesioned_random, grid_shape, axs=axs[1, :])
axs[0, 0].set_ylabel("Ranked lesioning", fontsize=20)
axs[1, 0].set_ylabel("Random lesioning", fontsize=20)
plt.show()

# R(t) comparison
R_base, _ = get_R(res_base["theta"])
R_ranked, _ = get_R(res_ranked_lesion["theta"])
ts_plot = res_base["ts"]

fig, ax = plot_single_metric_rt_comparison(
    ts_plot, R_base, R_ranked, R_random_lesion_mean, R_random_lesion_std,
    lesion_frac=lesion_frac,
    metric_name=metric_name,
)
plt.show()

In [ ]:
lesion_fracs = np.arange(0, 0.3, 0.02)

single_metric_results = run_lesion_study(
    sim,
    metric,
    T_END=T_END,
    dt=dt,
    base_seed=SEED,
    n_random_repeats=N_RANDOM_REPEATS,
    lesion_fracs=lesion_fracs,
    lesion_strength=lesion_strength,
)

fig, axes = plot_r_vs_lesion_frac(
    lesion_fracs,
    single_metric_results["R_final_ranked"],
    single_metric_results["R_avg_ranked"],
    single_metric_results["R_final_random"],
    single_metric_results["R_avg_random"],
    title=f"R vs degradation: {metric_name}",
)
plt.show()

In [ ]:
# AUC / ABC from the same run_lesion_study output
AUC_ranked = single_metric_results["AUC_ranked"]
AUC_random = single_metric_results["AUC_random"]
ABC = single_metric_results["ABC"]

print(f"AUC_ranked: {AUC_ranked:.4f}")
print(f"AUC_random: {AUC_random:.4f}")
print(f"ABC:        {ABC:.4f}")

fig, ax = plot_auc_abc_diagram(
    lesion_fracs,
    single_metric_results["R_avg_ranked"],
    single_metric_results["R_avg_random"],
    AUC_ranked=AUC_ranked,
    AUC_random=AUC_random,
    ABC=ABC,
    title=f"AUC / ABC geometry: {metric_name}",
)
plt.show()

### Evaluate on all metrics

In [ ]:
# Run lesion study for each metric (same logic as run_lesion_study / evaluate_metric_scores lesion loop)
metric_scores = {}
for metric_name, metric in metrics.items():
    print(f"Evaluating {metric_name}...")
    metric_scores[metric_name] = run_lesion_study(
        sim,
        metric,
        T_END=T_END,
        dt=dt,
        base_seed=SEED,
        n_random_repeats=N_RANDOM_REPEATS,
        lesion_fracs=lesion_fracs,
        lesion_strength=lesion_strength,
    )

In [ ]:
fig, ax = plot_auc_dotplot(metric_scores, title="R AUC: importance-ranked vs random node lesions")
plt.show()

fig, ax = plot_abc_lollipop(metric_scores, title="Area between curves (higher = better targeting)")
plt.show()

## Multi-seed ensemble

Run the same lesion study over multiple seeds to quantify how much ABC scores vary due to random initialisation.

In [ ]:
N_SEEDS_DEMO = 5
BASE_SEED_DEMO = 42
demo_seeds = [BASE_SEED_DEMO + i for i in range(N_SEEDS_DEMO)]

demo_results: dict[str, dict[int, dict]] = {"demo": {}}

for seed in demo_seeds:
    print(f"  seed={seed}...")
    sim_s = build_simulation(config=cfg, rng=np.random.default_rng(seed))
    sim_s.run((0, T_END), dt)
    demo_results["demo"][seed] = evaluate_metric_scores(
        sim_s,
        T_END=T_END,
        dt=dt,
        RNG=np.random.default_rng(seed),
        n_random_repeats=N_RANDOM_REPEATS,
        lesion_fracs=lesion_fracs,
        lesion_strength=lesion_strength,
        verbose=False,
        base_seed=seed,
    )

print(f"\nDone – {N_SEEDS_DEMO} seeds.")

# Aggregate and plot
demo_agg = aggregate_scores(demo_results)
metrics_demo = demo_agg["demo"]["metrics"]
abc_by_metric = {
    m: [demo_results["demo"][s][m]["ABC"] for s in demo_seeds]
    for m in metrics_demo
}

fig, ax = plot_abc_spread_boxplot_h(
    abc_by_metric,
    title=f"ABC spread across {N_SEEDS_DEMO} seeds",
    patch_alpha=0.8,
)
plt.show()